# 商品カテゴリ別7日間売上予測MLパイプライン

このノートブックでは、Snowflake MLを使用して以下のEnd-to-End機械学習パイプラインを構築します：

1. **Compute Pool作成**: コンテナランタイム環境の準備
2. **データ準備**: POSトランザクションデータと商品マスタの結合・集計
3. **特徴量エンジニアリング**: 時系列特徴量の作成とFeature Storeへの登録
4. **モデル学習**: カテゴリを特徴量として投入した単一モデルの訓練
5. **実験トラッキング**: Snowflake Experiment Trackingでモデル管理
6. **モデル登録**: Snowflake Model RegistryへSQLから推論可能な形で登録
7. **ML Observability**: Model Monitorによる継続監視の設定
8. **評価・可視化**: カテゴリ別予測精度のRMSE/MAPE評価

### データソース
- `FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS`: POSトランザクションデータ
- `FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER`: 商品マスタ

---
## 0. 事前準備: Compute Poolの作成

**Snowflake ML Observability（Model Monitor）やSPCS推論サービス**を使用するためには、Compute Poolが必要です。
Compute Poolは、Snowpark Container Services (SPCS) でコンテナを実行するための計算リソースプールです。

In [ ]:
from snowflake.snowpark import Session
import os

connection_params = {
    "connection_name": os.getenv("SNOWFLAKE_CONNECTION_NAME") or "pm"
}
session = Session.builder.configs(connection_params).create()
print(f"接続成功: {session.get_current_account()}")
print(f"現在のロール: {session.get_current_role()}")
print(f"現在のウェアハウス: {session.get_current_warehouse()}")

In [ ]:
COMPUTE_POOL_NAME = "SALES_FORECAST_POOL"

create_pool_sql = f"""
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL_NAME}
    MIN_NODES = 1
    MAX_NODES = 3
    INSTANCE_FAMILY = CPU_X64_S
    AUTO_SUSPEND_SECS = 3600
    AUTO_RESUME = TRUE
    COMMENT = 'Sales forecast ML pipeline compute pool'
"""

session.sql(create_pool_sql).collect()
print(f"Compute Pool '{COMPUTE_POOL_NAME}' を作成/確認しました")

pool_status = session.sql(f"SHOW COMPUTE POOLS LIKE '{COMPUTE_POOL_NAME}'").collect()
print(f"ステータス: {pool_status[0]['state']}")

---
## 1. データ準備と探索

### Snowflake MLにおけるデータアクセス
Snowpark DataFrameを使用して、データをSnowflake上で効率的に処理します。
大量データでもSnowflakeのコンピューティングパワーを活用してプッシュダウン処理を行います。

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd
import numpy as np

DATABASE = "FOODEX_DEMO"
SCHEMA = "BUYER_AGENT"
session.use_database(DATABASE)
session.use_schema(SCHEMA)

transactions_df = session.table("ID_POS_TRANSACTIONS")
products_df = session.table("PRODUCT_MASTER")

print(f"トランザクション件数: {transactions_df.count():,}")
print(f"商品マスタ件数: {products_df.count():,}")

In [ ]:
date_range = transactions_df.select(
    F.min("TRANSACTION_DATE").alias("MIN_DATE"),
    F.max("TRANSACTION_DATE").alias("MAX_DATE")
).collect()[0]

print(f"データ期間: {date_range['MIN_DATE']} ~ {date_range['MAX_DATE']}")

In [ ]:
category_stats = session.sql("""
    SELECT 
        pm.CATEGORY_MEDIUM,
        COUNT(DISTINCT t.TRANSACTION_DATE) as DAYS,
        SUM(t.SALES_AMOUNT) as TOTAL_SALES,
        COUNT(*) as TRANSACTION_COUNT
    FROM ID_POS_TRANSACTIONS t
    JOIN PRODUCT_MASTER pm ON t.PRODUCT_ID = pm.PRODUCT_ID
    GROUP BY pm.CATEGORY_MEDIUM
    ORDER BY TOTAL_SALES DESC
""").to_pandas()

print(f"カテゴリ数: {len(category_stats)}")
category_stats.head(10)

---
## 2. 日次・カテゴリ別売上集計

予測対象となる日次×カテゴリ別の売上データを作成します。

In [ ]:
daily_sales_sql = """
CREATE OR REPLACE TABLE DAILY_CATEGORY_SALES AS
SELECT 
    t.TRANSACTION_DATE::DATE as SALES_DATE,
    pm.CATEGORY_MEDIUM,
    SUM(t.SALES_AMOUNT) as DAILY_SALES,
    COUNT(*) as TRANSACTION_COUNT,
    COUNT(DISTINCT t.CUSTOMER_ID) as UNIQUE_CUSTOMERS,
    SUM(t.QUANTITY) as TOTAL_QUANTITY,
    AVG(t.UNIT_SELLING_PRICE) as AVG_UNIT_PRICE,
    DAYOFWEEK(t.TRANSACTION_DATE) as DAY_OF_WEEK,
    DAYOFMONTH(t.TRANSACTION_DATE) as DAY_OF_MONTH,
    MONTH(t.TRANSACTION_DATE) as MONTH,
    CASE WHEN DAYOFWEEK(t.TRANSACTION_DATE) IN (0, 6) THEN 1 ELSE 0 END as IS_WEEKEND
FROM ID_POS_TRANSACTIONS t
JOIN PRODUCT_MASTER pm ON t.PRODUCT_ID = pm.PRODUCT_ID
GROUP BY 
    t.TRANSACTION_DATE::DATE,
    pm.CATEGORY_MEDIUM
ORDER BY SALES_DATE, CATEGORY_MEDIUM
"""

session.sql(daily_sales_sql).collect()
print("日次カテゴリ別売上テーブルを作成しました")

daily_sales_df = session.table("DAILY_CATEGORY_SALES")
print(f"レコード数: {daily_sales_df.count():,}")
daily_sales_df.show(10)

---
## 3. 特徴量エンジニアリング & Feature Store登録

### Snowflake Feature Storeとは
Feature Storeは、ML特徴量を一元管理するための仕組みです：
- **再利用性**: 特徴量を複数のモデルで共有
- **一貫性**: 学習時と推論時で同じ特徴量ロジックを使用
- **自動更新**: Dynamic Tableによる特徴量の自動リフレッシュ
- **バージョン管理**: 特徴量のバージョニング

### 作成する特徴量
- ラグ特徴量（1日前〜7日前の売上）
- 移動平均特徴量（7日、14日、28日）
- 曜日・月の周期性特徴量

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity

FEATURE_STORE_SCHEMA = "BUYER_AGENT"

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=FEATURE_STORE_SCHEMA,
    default_warehouse=session.get_current_warehouse()
)
print(f"Feature Store接続: {DATABASE}.{FEATURE_STORE_SCHEMA}")

In [ ]:
category_date_entity = Entity(
    name="CATEGORY_DATE",
    join_keys=["CATEGORY_MEDIUM", "SALES_DATE"],
    desc="商品カテゴリと日付の複合エンティティ"
)

fs.register_entity(category_date_entity)
print("エンティティを登録しました")

In [ ]:
feature_sql = """
SELECT 
    SALES_DATE,
    CATEGORY_MEDIUM,
    DAILY_SALES,
    TRANSACTION_COUNT,
    UNIQUE_CUSTOMERS,
    TOTAL_QUANTITY,
    AVG_UNIT_PRICE,
    DAY_OF_WEEK,
    DAY_OF_MONTH,
    MONTH,
    IS_WEEKEND,
    
    -- ラグ特徴量（過去の売上）
    LAG(DAILY_SALES, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_LAG_1,
    LAG(DAILY_SALES, 2) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_LAG_2,
    LAG(DAILY_SALES, 3) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_LAG_3,
    LAG(DAILY_SALES, 7) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_LAG_7,
    LAG(DAILY_SALES, 14) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_LAG_14,
    LAG(DAILY_SALES, 28) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_LAG_28,
    
    -- 移動平均特徴量
    AVG(DAILY_SALES) OVER (
        PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE 
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as SALES_MA_7,
    AVG(DAILY_SALES) OVER (
        PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE 
        ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
    ) as SALES_MA_14,
    AVG(DAILY_SALES) OVER (
        PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE 
        ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING
    ) as SALES_MA_28,
    
    -- 移動標準偏差（ボラティリティ）
    STDDEV(DAILY_SALES) OVER (
        PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE 
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as SALES_STD_7,
    
    -- トレンド特徴量（7日前との比較）
    DAILY_SALES - LAG(DAILY_SALES, 7) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) as SALES_DIFF_7,
    
    -- 曜日別平均との比較
    AVG(DAILY_SALES) OVER (
        PARTITION BY CATEGORY_MEDIUM, DAY_OF_WEEK ORDER BY SALES_DATE 
        ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING
    ) as SAME_DOW_AVG_4W
    
FROM DAILY_CATEGORY_SALES
"""

feature_df = session.sql(feature_sql)
print("特徴量DataFrameを作成しました")
feature_df.show(5)

In [ ]:
sales_feature_view = FeatureView(
    name="SALES_FORECAST_FEATURES",
    entities=[category_date_entity],
    feature_df=feature_df,
    timestamp_col="SALES_DATE",
    refresh_freq="1 day",
    desc="日次カテゴリ別売上予測のための特徴量セット"
)

sales_feature_view = sales_feature_view.attach_feature_desc({
    "DAILY_SALES": "当日の売上金額（予測ターゲット）",
    "SALES_LAG_1": "1日前の売上金額",
    "SALES_LAG_7": "7日前（同曜日）の売上金額",
    "SALES_MA_7": "過去7日間の売上移動平均",
    "SALES_MA_28": "過去28日間の売上移動平均",
    "SALES_STD_7": "過去7日間の売上標準偏差",
    "IS_WEEKEND": "週末フラグ（土日=1）",
    "SAME_DOW_AVG_4W": "過去4週間の同曜日平均売上"
})

print("Feature View定義を作成しました")

In [ ]:
try:
    fs.delete_feature_view("SALES_FORECAST_FEATURES", "V1")
    print("既存のFeature Viewを削除しました")
except:
    pass

registered_fv = fs.register_feature_view(
    feature_view=sales_feature_view,
    version="V1",
    block=True,
    overwrite=True
)

print(f"Feature View登録完了: {registered_fv.name} v{registered_fv.version}")

In [ ]:
print("登録済みFeature Views:")
fs.list_feature_views().show()

---
## 4. 学習データの準備

### Train/Test分割戦略
時系列データのため、**最後30日をテストデータ**として使用します。
これにより、将来の予測性能を正しく評価できます。

In [ ]:
retrieved_fv = fs.get_feature_view("SALES_FORECAST_FEATURES", "V1")

training_data_sql = """
SELECT * FROM FOODEX_DEMO.BUYER_AGENT.SALES_FORECAST_FEATURES$V1
WHERE SALES_LAG_28 IS NOT NULL
  AND SAME_DOW_AVG_4W IS NOT NULL
ORDER BY SALES_DATE, CATEGORY_MEDIUM
"""

full_data = session.sql(training_data_sql).to_pandas()
print(f"有効なデータ件数: {len(full_data):,}")
print(f"データ期間: {full_data['SALES_DATE'].min()} ~ {full_data['SALES_DATE'].max()}")

In [ ]:
from datetime import timedelta

max_date = pd.to_datetime(full_data['SALES_DATE'].max())
test_start_date = max_date - timedelta(days=29)

train_data = full_data[pd.to_datetime(full_data['SALES_DATE']) < test_start_date].copy()
test_data = full_data[pd.to_datetime(full_data['SALES_DATE']) >= test_start_date].copy()

print(f"学習データ: {len(train_data):,} 件 ({train_data['SALES_DATE'].min()} ~ {train_data['SALES_DATE'].max()})")
print(f"テストデータ: {len(test_data):,} 件 ({test_data['SALES_DATE'].min()} ~ {test_data['SALES_DATE'].max()})")

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
all_categories = full_data['CATEGORY_MEDIUM'].unique()
le.fit(all_categories)

train_data['CATEGORY_ENCODED'] = le.transform(train_data['CATEGORY_MEDIUM'])
test_data['CATEGORY_ENCODED'] = le.transform(test_data['CATEGORY_MEDIUM'])

print(f"カテゴリ数: {len(all_categories)}")
print(f"エンコーディング例: {dict(zip(all_categories[:5], le.transform(all_categories[:5])))}")

In [ ]:
FEATURE_COLS = [
    'CATEGORY_ENCODED',
    'TRANSACTION_COUNT', 'UNIQUE_CUSTOMERS', 'TOTAL_QUANTITY', 'AVG_UNIT_PRICE',
    'DAY_OF_WEEK', 'DAY_OF_MONTH', 'MONTH', 'IS_WEEKEND',
    'SALES_LAG_1', 'SALES_LAG_2', 'SALES_LAG_3', 'SALES_LAG_7', 'SALES_LAG_14', 'SALES_LAG_28',
    'SALES_MA_7', 'SALES_MA_14', 'SALES_MA_28',
    'SALES_STD_7', 'SALES_DIFF_7', 'SAME_DOW_AVG_4W'
]
TARGET_COL = 'DAILY_SALES'

X_train = train_data[FEATURE_COLS].fillna(0)
y_train = train_data[TARGET_COL]
X_test = test_data[FEATURE_COLS].fillna(0)
y_test = test_data[TARGET_COL]

print(f"特徴量数: {len(FEATURE_COLS)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

---
## 5. モデル学習 & 実験トラッキング

### Snowflake Experiment Trackingとは
Snowflake Experiment Trackingは、MLモデルの学習過程を記録・管理する機能です：
- **パラメータログ**: ハイパーパラメータの記録
- **メトリクスログ**: 評価指標の記録
- **モデルログ**: 学習済みモデルの保存
- **比較機能**: 複数の実験結果を比較

### モデル選択
LightGBMを使用します。理由：
- カテゴリ特徴量のネイティブサポート
- 高速な学習
- 時系列データでの良好な性能

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.model.model_signature import infer_signature

exp = ExperimentTracking(
    session=session,
    database_name=DATABASE,
    schema_name=SCHEMA
)

EXPERIMENT_NAME = "SALES_FORECAST_EXPERIMENT"
exp.set_experiment(EXPERIMENT_NAME)
print(f"実験名: {EXPERIMENT_NAME}")

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

params = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbosity': -1
}

RUN_NAME = "lgbm_sales_forecast_v1"

with exp.start_run(RUN_NAME):
    exp.log_params(params)
    exp.log_param("feature_count", len(FEATURE_COLS))
    exp.log_param("train_size", len(X_train))
    exp.log_param("test_size", len(X_test))
    
    print("モデル学習中...")
    model = LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric=['rmse', 'mape']
    )
    
    y_pred = model.predict(X_test)
    
    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    
    exp.log_metrics({
        "test_rmse": rmse,
        "test_mape": mape
    })
    
    sig = infer_signature(X_train, y_train)
    exp.log_model(
        model,
        model_name="SALES_FORECAST_MODEL",
        signatures={"predict": sig}
    )
    
    print(f"\n=== 全体評価結果 ===")
    print(f"RMSE: {rmse:,.2f}円")
    print(f"MAPE: {mape:.2f}%")

In [ ]:
session.sql(f"SHOW RUNS IN EXPERIMENT {DATABASE}.{SCHEMA}.{EXPERIMENT_NAME}").show()

---
## 6. カテゴリ別評価と可視化

各商品カテゴリ（CATEGORY_MEDIUM）ごとの予測精度を評価します。

In [ ]:
test_data['PREDICTED_SALES'] = y_pred

category_metrics = []
for category in test_data['CATEGORY_MEDIUM'].unique():
    cat_data = test_data[test_data['CATEGORY_MEDIUM'] == category]
    cat_rmse = root_mean_squared_error(cat_data['DAILY_SALES'], cat_data['PREDICTED_SALES'])
    cat_mape = mean_absolute_percentage_error(cat_data['DAILY_SALES'], cat_data['PREDICTED_SALES']) * 100
    category_metrics.append({
        'CATEGORY': category,
        'RMSE': cat_rmse,
        'MAPE': cat_mape,
        'AVG_SALES': cat_data['DAILY_SALES'].mean(),
        'SAMPLE_COUNT': len(cat_data)
    })

metrics_df = pd.DataFrame(category_metrics).sort_values('RMSE', ascending=False)
print("=== カテゴリ別予測精度 ===")
metrics_df.head(20)

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_20 = metrics_df.head(20)
colors = plt.cm.RdYlGn_r(top_20['MAPE'] / top_20['MAPE'].max())

axes[0].barh(range(len(top_20)), top_20['RMSE'], color=colors)
axes[0].set_yticks(range(len(top_20)))
axes[0].set_yticklabels(top_20['CATEGORY'])
axes[0].set_xlabel('RMSE (円)')
axes[0].set_title('カテゴリ別RMSE (Top 20)')
axes[0].invert_yaxis()

axes[1].barh(range(len(top_20)), top_20['MAPE'], color=colors)
axes[1].set_yticks(range(len(top_20)))
axes[1].set_yticklabels(top_20['CATEGORY'])
axes[1].set_xlabel('MAPE (%)')
axes[1].set_title('カテゴリ別MAPE (Top 20)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('category_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n=== サマリー ===")
print(f"全体RMSE: {rmse:,.2f}円")
print(f"全体MAPE: {mape:.2f}%")
print(f"カテゴリ平均RMSE: {metrics_df['RMSE'].mean():,.2f}円")
print(f"カテゴリ平均MAPE: {metrics_df['MAPE'].mean():.2f}%")

In [ ]:
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(range(len(feature_importance)), feature_importance['importance'])
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Feature Importance')
plt.title('特徴量重要度')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Snowflake Model Registryへの登録

### Model Registryとは
Snowflake Model Registryは、MLモデルを一元管理するための機能です：
- **バージョン管理**: モデルの複数バージョンを管理
- **SQL推論**: SQLクエリから直接モデルを呼び出し可能
- **SPCS推論**: REST APIエンドポイントとしてデプロイ可能
- **メタデータ管理**: モデルの説明、メトリクスを保存

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(
    session=session,
    database_name=DATABASE,
    schema_name=SCHEMA
)

MODEL_NAME = "SALES_FORECAST_MODEL"
VERSION_NAME = "V1"

existing_models = session.sql(f"SHOW MODELS LIKE '{MODEL_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
if len(existing_models) > 0:
    print(f"既存モデル発見: {MODEL_NAME}")
    try:
        reg.delete_model(MODEL_NAME)
        print("既存モデルを削除しました")
    except:
        pass

In [ ]:
sample_input = X_train.head(10)

mv = reg.log_model(
    model,
    model_name=MODEL_NAME,
    version_name=VERSION_NAME,
    sample_input_data=sample_input,
    conda_dependencies=["lightgbm", "scikit-learn"],
    target_platforms=["WAREHOUSE"],
    comment=f"商品カテゴリ別7日間売上予測モデル (RMSE: {rmse:.2f}, MAPE: {mape:.2f}%)",
    metrics={
        "test_rmse": rmse,
        "test_mape": mape,
        "train_size": len(X_train),
        "test_size": len(X_test)
    }
)

print(f"\nモデル登録完了: {mv.model_name} version {mv.version_name}")

In [ ]:
print("=== 登録済みモデル一覧 ===")
session.sql(f"SHOW MODELS IN SCHEMA {DATABASE}.{SCHEMA}").show()

print("\n=== モデル関数 ===")
session.sql(f"SHOW FUNCTIONS IN MODEL {DATABASE}.{SCHEMA}.{MODEL_NAME}").show()

---
## 8. SQLからの推論テスト

登録したモデルをSQLから呼び出して推論を実行します。
これにより、データパイプラインや他のアプリケーションから直接予測を取得できます。

In [ ]:
inference_sql = f"""
WITH test_features AS (
    SELECT * FROM {DATABASE}.{SCHEMA}.SALES_FORECAST_FEATURES$V1
    WHERE SALES_DATE >= DATEADD(day, -7, CURRENT_DATE())
      AND SALES_LAG_28 IS NOT NULL
    LIMIT 10
)
SELECT 
    SALES_DATE,
    CATEGORY_MEDIUM,
    DAILY_SALES as ACTUAL_SALES,
    MODEL({DATABASE}.{SCHEMA}.{MODEL_NAME}, {VERSION_NAME})!PREDICT(
        0 as CATEGORY_ENCODED,
        TRANSACTION_COUNT,
        UNIQUE_CUSTOMERS,
        TOTAL_QUANTITY,
        AVG_UNIT_PRICE,
        DAY_OF_WEEK,
        DAY_OF_MONTH,
        MONTH,
        IS_WEEKEND,
        SALES_LAG_1,
        SALES_LAG_2,
        SALES_LAG_3,
        SALES_LAG_7,
        SALES_LAG_14,
        SALES_LAG_28,
        SALES_MA_7,
        SALES_MA_14,
        SALES_MA_28,
        SALES_STD_7,
        SALES_DIFF_7,
        SAME_DOW_AVG_4W
    ):output_feature_0::FLOAT as PREDICTED_SALES
FROM test_features
"""

print("=== SQL推論テスト ===")
session.sql(inference_sql).show()

---
## 9. Model Monitor設定（ML Observability）

### Model Monitorとは
Model Monitorは、本番環境でのモデル性能を継続的に監視する機能です：
- **ドリフト検出**: 特徴量分布の変化を検出（PSI、KLダイバージェンス）
- **性能監視**: RMSE、MAPEなどの指標を継続的に追跡
- **セグメント別分析**: カテゴリ別の性能を個別に監視
- **アラート**: 閾値を超えた場合の通知

### 監視データの準備
推論ログと実績売上を保存するテーブルを作成します。

In [ ]:
monitor_table_sql = f"""
CREATE OR REPLACE TABLE {DATABASE}.{SCHEMA}.SALES_PREDICTIONS_LOG (
    PREDICTION_ID VARCHAR(36) DEFAULT UUID_STRING(),
    PREDICTION_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    SALES_DATE DATE,
    CATEGORY_MEDIUM VARCHAR(100),
    
    -- 特徴量
    CATEGORY_ENCODED INTEGER,
    TRANSACTION_COUNT INTEGER,
    UNIQUE_CUSTOMERS INTEGER,
    TOTAL_QUANTITY INTEGER,
    AVG_UNIT_PRICE FLOAT,
    DAY_OF_WEEK INTEGER,
    DAY_OF_MONTH INTEGER,
    MONTH INTEGER,
    IS_WEEKEND INTEGER,
    SALES_LAG_1 FLOAT,
    SALES_LAG_7 FLOAT,
    SALES_MA_7 FLOAT,
    SALES_MA_28 FLOAT,
    
    -- 予測結果
    PREDICTED_SALES FLOAT,
    
    -- 実績（後から更新）
    ACTUAL_SALES FLOAT
)
"""

session.sql(monitor_table_sql).collect()
print("推論ログテーブルを作成しました")

In [ ]:
insert_log_sql = f"""
INSERT INTO {DATABASE}.{SCHEMA}.SALES_PREDICTIONS_LOG (
    SALES_DATE, CATEGORY_MEDIUM, CATEGORY_ENCODED,
    TRANSACTION_COUNT, UNIQUE_CUSTOMERS, TOTAL_QUANTITY, AVG_UNIT_PRICE,
    DAY_OF_WEEK, DAY_OF_MONTH, MONTH, IS_WEEKEND,
    SALES_LAG_1, SALES_LAG_7, SALES_MA_7, SALES_MA_28,
    PREDICTED_SALES, ACTUAL_SALES
)
SELECT 
    SALES_DATE,
    CATEGORY_MEDIUM,
    0 as CATEGORY_ENCODED,
    TRANSACTION_COUNT, UNIQUE_CUSTOMERS, TOTAL_QUANTITY, AVG_UNIT_PRICE,
    DAY_OF_WEEK, DAY_OF_MONTH, MONTH, IS_WEEKEND,
    SALES_LAG_1, SALES_LAG_7, SALES_MA_7, SALES_MA_28,
    MODEL({DATABASE}.{SCHEMA}.{MODEL_NAME}, {VERSION_NAME})!PREDICT(
        0, TRANSACTION_COUNT, UNIQUE_CUSTOMERS, TOTAL_QUANTITY, AVG_UNIT_PRICE,
        DAY_OF_WEEK, DAY_OF_MONTH, MONTH, IS_WEEKEND,
        SALES_LAG_1, SALES_LAG_2, SALES_LAG_3, SALES_LAG_7, SALES_LAG_14, SALES_LAG_28,
        SALES_MA_7, SALES_MA_14, SALES_MA_28,
        SALES_STD_7, SALES_DIFF_7, SAME_DOW_AVG_4W
    ):output_feature_0::FLOAT as PREDICTED_SALES,
    DAILY_SALES as ACTUAL_SALES
FROM {DATABASE}.{SCHEMA}.SALES_FORECAST_FEATURES$V1
WHERE SALES_DATE >= DATEADD(day, -30, CURRENT_DATE())
  AND SALES_LAG_28 IS NOT NULL
"""

session.sql(insert_log_sql).collect()
log_count = session.sql(f"SELECT COUNT(*) as CNT FROM {DATABASE}.{SCHEMA}.SALES_PREDICTIONS_LOG").collect()[0]['CNT']
print(f"推論ログを挿入しました: {log_count:,} 件")

In [ ]:
MONITOR_NAME = "SALES_FORECAST_MONITOR"

try:
    session.sql(f"DROP MODEL MONITOR IF EXISTS {DATABASE}.{SCHEMA}.{MONITOR_NAME}").collect()
    print("既存のModel Monitorを削除しました")
except:
    pass

create_monitor_sql = f"""
CREATE MODEL MONITOR {DATABASE}.{SCHEMA}.{MONITOR_NAME} WITH
    MODEL = {DATABASE}.{SCHEMA}.{MODEL_NAME}
    VERSION = '{VERSION_NAME}'
    FUNCTION = 'PREDICT'
    SOURCE = {DATABASE}.{SCHEMA}.SALES_PREDICTIONS_LOG
    WAREHOUSE = {session.get_current_warehouse()}
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTED_SALES')
    ACTUAL_SCORE_COLUMNS = ('ACTUAL_SALES')
    SEGMENT_COLUMNS = ('CATEGORY_MEDIUM')
"""

session.sql(create_monitor_sql).collect()
print(f"Model Monitor '{MONITOR_NAME}' を作成しました")

In [ ]:
print("=== Model Monitor状態 ===")
session.sql(f"DESC MODEL MONITOR {DATABASE}.{SCHEMA}.{MONITOR_NAME}").show()

---
## 10. Model Monitor メトリクスのクエリ

Model Monitorから性能メトリクスとドリフトメトリクスを取得します。

In [ ]:
performance_sql = f"""
SELECT * 
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    '{DATABASE}.{SCHEMA}.{MONITOR_NAME}',
    'RMSE',
    'DAY',
    DATEADD(day, -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))
ORDER BY WINDOW_END DESC
"""

print("=== 性能メトリクス (RMSE) ===")
try:
    session.sql(performance_sql).show()
except Exception as e:
    print(f"注意: メトリクスはMonitorの初回リフレッシュ後に利用可能になります")
    print(f"エラー: {e}")

In [ ]:
segment_metrics_sql = f"""
SELECT * 
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    '{DATABASE}.{SCHEMA}.{MONITOR_NAME}',
    'RMSE',
    'DAY',
    DATEADD(day, -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
    '{{"SEGMENTS": [{{"column": "CATEGORY_MEDIUM", "value": "シリアル"}}]}}'
))
"""

print("=== カテゴリ別メトリクス例 (シリアル) ===")
try:
    session.sql(segment_metrics_sql).show()
except Exception as e:
    print(f"注意: セグメント別メトリクスはMonitorの初回リフレッシュ後に利用可能になります")

---
## 11. カテゴリ別監視ダッシュボード用データ

カテゴリ別のRMSE/MAPEを継続的に可視化するためのビューを作成します。

In [ ]:
category_monitor_view_sql = f"""
CREATE OR REPLACE VIEW {DATABASE}.{SCHEMA}.SALES_FORECAST_CATEGORY_METRICS AS
SELECT 
    DATE_TRUNC('day', PREDICTION_TIMESTAMP) as METRIC_DATE,
    CATEGORY_MEDIUM,
    COUNT(*) as PREDICTION_COUNT,
    AVG(ACTUAL_SALES) as AVG_ACTUAL_SALES,
    AVG(PREDICTED_SALES) as AVG_PREDICTED_SALES,
    SQRT(AVG(POWER(ACTUAL_SALES - PREDICTED_SALES, 2))) as RMSE,
    AVG(ABS(ACTUAL_SALES - PREDICTED_SALES) / NULLIF(ACTUAL_SALES, 0)) * 100 as MAPE,
    CORR(ACTUAL_SALES, PREDICTED_SALES) as CORRELATION
FROM {DATABASE}.{SCHEMA}.SALES_PREDICTIONS_LOG
WHERE ACTUAL_SALES IS NOT NULL
GROUP BY DATE_TRUNC('day', PREDICTION_TIMESTAMP), CATEGORY_MEDIUM
ORDER BY METRIC_DATE DESC, CATEGORY_MEDIUM
"""

session.sql(category_monitor_view_sql).collect()
print("カテゴリ別メトリクスビューを作成しました")

session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.SALES_FORECAST_CATEGORY_METRICS LIMIT 20").show()

In [ ]:
category_metrics_data = session.sql(f"""
    SELECT * FROM {DATABASE}.{SCHEMA}.SALES_FORECAST_CATEGORY_METRICS
    ORDER BY RMSE DESC
""").to_pandas()

if len(category_metrics_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    top_cats = category_metrics_data.groupby('CATEGORY_MEDIUM').agg({
        'RMSE': 'mean',
        'MAPE': 'mean'
    }).sort_values('RMSE', ascending=False).head(15)
    
    colors = plt.cm.RdYlGn_r(top_cats['MAPE'] / top_cats['MAPE'].max())
    
    axes[0].barh(range(len(top_cats)), top_cats['RMSE'], color=colors)
    axes[0].set_yticks(range(len(top_cats)))
    axes[0].set_yticklabels(top_cats.index)
    axes[0].set_xlabel('RMSE (円)')
    axes[0].set_title('監視データ: カテゴリ別RMSE')
    axes[0].invert_yaxis()
    
    axes[1].barh(range(len(top_cats)), top_cats['MAPE'], color=colors)
    axes[1].set_yticks(range(len(top_cats)))
    axes[1].set_yticklabels(top_cats.index)
    axes[1].set_xlabel('MAPE (%)')
    axes[1].set_title('監視データ: カテゴリ別MAPE')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('monitoring_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 12. 最終サマリー

パイプライン全体のまとめと、作成されたリソース一覧です。

In [ ]:
print("="*60)
print("    売上予測MLパイプライン 構築完了サマリー")
print("="*60)

print("\n【1. Compute Pool】")
print(f"   名前: {COMPUTE_POOL_NAME}")
print(f"   インスタンス: CPU_X64_S (1-3ノード)")

print("\n【2. Feature Store】")
print(f"   Feature View: SALES_FORECAST_FEATURES (V1)")
print(f"   Entity: CATEGORY_DATE")
print(f"   リフレッシュ頻度: 1日")

print("\n【3. モデル】")
print(f"   アルゴリズム: LightGBM Regressor")
print(f"   特徴量数: {len(FEATURE_COLS)}")
print(f"   学習データ: {len(X_train):,} 件")
print(f"   テストデータ: {len(X_test):,} 件")

print("\n【4. 評価結果】")
print(f"   全体RMSE: {rmse:,.2f}円")
print(f"   全体MAPE: {mape:.2f}%")

print("\n【5. Experiment Tracking】")
print(f"   実験名: {EXPERIMENT_NAME}")
print(f"   Run名: {RUN_NAME}")

print("\n【6. Model Registry】")
print(f"   モデル名: {MODEL_NAME}")
print(f"   バージョン: {VERSION_NAME}")
print(f"   ターゲット: WAREHOUSE (SQL推論可能)")

print("\n【7. Model Monitor】")
print(f"   モニター名: {MONITOR_NAME}")
print(f"   リフレッシュ頻度: 1日")
print(f"   セグメント列: CATEGORY_MEDIUM")
print(f"   監視メトリクス: RMSE, MAPE, ドリフト(PSI)")

print("\n【8. 作成されたテーブル/ビュー】")
print(f"   - {DATABASE}.{SCHEMA}.DAILY_CATEGORY_SALES")
print(f"   - {DATABASE}.{SCHEMA}.SALES_FORECAST_FEATURES$V1")
print(f"   - {DATABASE}.{SCHEMA}.SALES_PREDICTIONS_LOG")
print(f"   - {DATABASE}.{SCHEMA}.SALES_FORECAST_CATEGORY_METRICS")

print("\n" + "="*60)
print("   SQL推論例:")
print("="*60)
print(f"""
SELECT 
    MODEL({DATABASE}.{SCHEMA}.{MODEL_NAME}, {VERSION_NAME})!PREDICT(
        <features...>
    ):output_feature_0::FLOAT as PREDICTED_SALES
FROM your_table;
""")

In [ ]:
session.close()
print("セッションをクローズしました")